# Train Tokenizer

Colab-friendly notebook for training the SentencePiece BPE tokenizer and encoding the training corpus.

## Colab Setup

If you opened this notebook from GitHub in Colab, run the setup cell. It installs dependencies and keeps paths relative to the repository.

In [ ]:
%pip install -q sentencepiece numpy

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "pretrain":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pretrain.tokenizer import Tokenizer

## Configuration

In [ ]:
CORPUS = PROJECT_ROOT / "data" / "van-life-story.txt"
VOCAB_SIZE = 1024
OUTPUT_DIR = PROJECT_ROOT / "pretrain" / "data"
OUTPUT_FORMAT = "bin"  # "bin" or "jsonl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"corpus: {CORPUS}")
print(f"output: {OUTPUT_DIR}")

## Train

In [ ]:
tok = Tokenizer.train(
    text_path=CORPUS,
    vocab_size=VOCAB_SIZE,
    output_dir=OUTPUT_DIR,
    name="sp",
)

print(f"vocab size: {tok.vocab_size:,}")

## Encode Corpus

In [ ]:
corpus_stem = CORPUS.stem
ext = "bin" if OUTPUT_FORMAT == "bin" else "jsonl"
out_path = OUTPUT_DIR / f"train.{corpus_stem}.{ext}"

if OUTPUT_FORMAT == "jsonl":
    n_tokens = 0
    with open(CORPUS, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            line = line.rstrip("\n")
            if not line:
                continue
            tokens = tok.encode(line)
            fout.write(json.dumps(tokens) + "\n")
            n_tokens += len(tokens)
elif OUTPUT_FORMAT == "bin":
    all_tokens: list[int] = []
    with open(CORPUS, "r", encoding="utf-8") as fin:
        for line in fin:
            line = line.rstrip("\n")
            if not line:
                continue
            all_tokens.extend(tok.encode(line))
    n_tokens = len(all_tokens)
    arr = np.array(all_tokens, dtype=np.uint32)
    arr.tofile(out_path)
else:
    raise ValueError(f"Unsupported OUTPUT_FORMAT: {OUTPUT_FORMAT}")

print("[done] tokenizer trained and corpus encoded.")
print(f"model  : {tok.model_path}")
print(f"corpus : {out_path} ({n_tokens:,} tokens)")

## Sanity Check

In [ ]:
sample = "The van rolled toward the coast."
ids = tok.encode(sample)
print(ids)
print(tok.decode(ids))